# Lab 2: a multi-agent research system

Scenario 3, in two stages.

A **research agent** fans out to four searchers at once: two fields, visual art and music, on
two channels, one published and one internal. Then its findings are **chained into a formatter
agent**, a second run that turns them into a three-section report. That is the whole lab.

The chain is the part worth watching. The formatter is a separate run with an empty context,
so the only thing it will ever know about the research is what the prompt carries into it. A
subagent works the same way, and here it is visible rather than asserted.

The corpus is three documents, rigged so that each section of the report fills from a
different place:

| Section | What fills it |
|---|---|
| Well established | Visual art, where the published source stands on its own |
| Contested | Music, where the two channels report different figures, because they counted different populations over windows that do not overlap |
| Coverage gaps | The internal channel holds nothing on visual art, so that search runs and matches nothing |

Sections 1 to 4 make no API call. Sections 5 and 6 are one run each, both capped in turns and
in money, both on the smaller model.

Setup, once, in the `code/` directory above this one: copy `.env.example` to `.env` and read
the notes at the top of it. On a Claude subscription you leave the credential lines blank and
run `claude` once to sign in; on API billing you put a key in `ANTHROPIC_API_KEY`. Every lab
reads that same file, and the first cell prints which of the two it is about to use.

## 1. The brief

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

LAB = Path.cwd()
CODE = LAB.parent
OUT = LAB / "out"

if not (CODE / "pyproject.toml").exists():
    raise SystemExit(
        f"Run this notebook from its own folder inside labs/code. The working "
        f"directory is {LAB}, and {CODE / 'pyproject.toml'} is not there. "
        f"In JupyterLab the working directory follows the notebook, so open it "
        f"from the file browser rather than starting the kernel elsewhere."
    )

ENV_FILE = CODE / ".env"
if not ENV_FILE.exists():
    raise SystemExit(f"No {ENV_FILE}. Copy .env.example to .env and read the notes at the top of it.")

load_dotenv(ENV_FILE)

# A name left blank in .env still reaches the environment, as an empty string. An empty
# ANTHROPIC_API_KEY fails the run rather than falling through to a login, so drop the blanks
# and let the credential chosen below be one that is really set.
for name in ("ANTHROPIC_API_KEY", "CLAUDE_CODE_OAUTH_TOKEN"):
    if not os.environ.get(name):
        os.environ.pop(name, None)

# The Agent SDK runs the claude binary rather than calling the API itself, so it takes the
# credential that binary takes, in that binary's order of preference: key first, then token,
# then the login claude saved when you signed in.
if os.environ.get("ANTHROPIC_API_KEY"):
    CREDENTIAL = "ANTHROPIC_API_KEY from .env, billed to your API account"
elif os.environ.get("CLAUDE_CODE_OAUTH_TOKEN"):
    CREDENTIAL = "CLAUDE_CODE_OAUTH_TOKEN from .env, drawn from your subscription"
else:
    CREDENTIAL = "the login claude saved, drawn from your subscription"

if not os.environ.get("LAB_MODEL_SMALL"):
    raise SystemExit(f"{ENV_FILE} has no value for LAB_MODEL_SMALL.")
MODEL = os.environ["LAB_MODEL_SMALL"]
OUT.mkdir(parents=True, exist_ok=True)

# The brief. Not a list of steps: a goal, the fields that must be covered, and the criteria a
# finished report has to meet.
BRIEF = {
    "topic": "AI in the creative industries",
    "goal": "Survey how AI is changing the creative industries across both fields below.",
    "fields": [{"id": "visual_art", "label": "visual art and design"},
               {"id": "music", "label": "music and audio"}],
    "criteria": [
        "Every finding carries a source URL, a verbatim excerpt, a publisher and the period "
        "the source measured.",
        "Where two sources disagree, both values are kept with their publisher and period. "
        "Do not choose between them and do not average them.",
        "A search that ran and matched nothing is reported as a gap with the field and the "
        "channel, not as a failure.",
    ],
}

print(f"model      from LAB_MODEL_SMALL in {ENV_FILE}")
print(f"credential {CREDENTIAL}")
print(f"fields     {', '.join(f['label'] for f in BRIEF['fields'])}")
print(f"output  {OUT}")

## 2. The corpus and the research server

Four tools over three documents, in process. `create_sdk_mcp_server` runs them inside this
Python process, so there is no subprocess and they read the corpus directly. The dict key the
server is registered under is what appears in the tool name, so registering it as
`{"research": server}` gives `mcp__research__search_web` and friends.

The two search tools are the whole of the tool-scoping lesson, and they are what make the two
searchers different agents rather than the same agent twice.

| Tool | Who gets it | Why it is separate |
|---|---|---|
| `search_web` | the web searcher | the published channel: what the sector says about itself |
| `search_internal` | the document searcher | the internal channel: what one organisation recorded |
| `load_document` | the document searcher | opens one document, and only a URL a search returned |
| `verify_fact` | the formatter | confirms one claim against one source, and nothing else |

`load_document` is the constrained replacement for a generic fetcher. A generic fetcher takes
any URL at all, including one an agent invented, and returns something plausible for it, which
is how a source nobody can check ends up cited in a report. This one refuses anything outside
the corpus and says what to do instead.

A refusal carries six fields, and each answers exactly one question the caller has to answer
anyway: `status` did it fail, `failure_type` is a retry worth trying, `attempted_query` what
should a retry change, `partial_results` what is already usable, `alternative_approaches`
which other route could work, `coverage_impact` what gap must the report annotate. That shape
is defined once, in `failed`, and nowhere else. `is_error` on the result is what marks a
failure on the wire: a tool that returns an error dictionary without it is reported as a
success, which is how a failure quietly becomes "nothing found".

The fixtures are short on purpose. They exist to give the tools something to serve, not to be
studied, and the retrieval is a set intersection rather than a scorer for the same reason.

In [ ]:
import re
from typing import Annotated, Any

from claude_agent_sdk import create_sdk_mcp_server, tool

# Three documents. The music pair report different figures, because one counted nineteen
# studios over eight months and excluded home studios, and the other counted every project in
# a quarter and included them. The internal channel holds nothing on visual art, which is what
# fills the coverage-gaps section.
DOCS = [
    {"field": "visual_art", "channel": "web",
     "url": "https://riverbend-design-review.example/2026/02/illustration-studios",
     "title": "Illustration studios and generated concept art",
     "publisher": "Riverbend Design Review",
     "publication_date": "2026-02-11", "collection_period": "2025-01 to 2025-09",
     "text": "Across forty-two studios, commissioning of external concept art fell by twelve "
             "per cent against the previous comparable period. The decline is concentrated "
             "in early-stage mood and layout work rather than in finished illustration, and "
             "studios reported that revision rounds rose over the same window."},
    {"field": "music", "channel": "web",
     "url": "https://riverbend-music-quarterly.example/2026/01/session-players",
     "title": "Session players and generated stems",
     "publisher": "Riverbend Music Quarterly",
     "publication_date": "2026-01-09", "collection_period": "2025-03 to 2025-10",
     "text": "Booking logs from nineteen recording studios show total session hours down by "
             "nine per cent against the previous comparable period. The decline is "
             "concentrated in library and advertising work rather than in album sessions. "
             "The analysis excludes home and project studios."},
    {"field": "music", "channel": "internal",
     "url": "https://intranet.example/post-production/2026/01/audio-post-note",
     "title": "Audio post production note",
     "publisher": "Post Production, internal",
     "publication_date": "2026-01-30", "collection_period": "Q4 2025",
     "text": "Session hours booked across the quarter fell by twenty-two per cent against "
             "the same quarter a year earlier. The review counts every project the "
             "department ran, including work recorded in home and project studios."},
]

# A search snippet is the opening of the document, so it is derived rather than stored twice.
for doc in DOCS:
    doc["excerpt"] = doc["text"].split(". ")[0] + "."

BY_URL = {d["url"]: d for d in DOCS}

# What a field is called when somebody asks about it. Matching whole words against this set
# keeps the lab about the technique rather than about retrieval quality.
FIELD_WORDS = {
    "visual_art": frozenset("visual art arts design designs illustration illustrations "
                            "concept gallery galleries studio studios".split()),
    "music": frozenset("music musical audio session sessions stem stems track tracks "
                       "recording recordings player players".split()),
}


def ok(payload):
    return {"content": [{"type": "text", "text": json.dumps(payload, ensure_ascii=False)}]}


def failed(kind, *, attempted, impact, alternatives, partial=None):
    # The six-field contract, defined here and nowhere else. is_error is what marks it a
    # failure on the wire; without it the caller reports the refusal as a success.
    payload = {"status": "partial_failure", "failure_type": kind,
               "attempted_query": attempted, "partial_results": partial or [],
               "alternative_approaches": alternatives, "coverage_impact": impact}
    return {"content": [{"type": "text", "text": json.dumps(payload, ensure_ascii=False)}],
            "is_error": True}


def results(query, channel):
    asked = set(re.findall(r"[a-z]+", query.lower()))
    hits = [{k: d[k] for k in ("title", "url", "publisher", "publication_date",
                               "collection_period", "excerpt")}
            for d in DOCS if d["channel"] == channel and asked & FIELD_WORDS[d["field"]]]
    return ok({"status": "success", "channel": channel, "attempted_query": query,
               "results": hits,
               "note": "" if hits else "The search ran and matched nothing on this channel."})


@tool(
    "search_web",
    "Search the published channel: trade research, sector monitors and official statistics "
    "about AI in the creative industries. Use it to survey what has been reported publicly "
    "about a field. Do not use it for one organisation own records (use search_internal). "
    "Every hit carries a title, a URL, a publisher, a publication date, the period measured "
    "and a verbatim excerpt. A search that matches nothing returns an empty list, which is "
    "an answer and not an error.",
    {"query": Annotated[str, "What to search for, in plain words."]},
)
async def search_web(args):
    return results(str(args.get("query", "")), "web")


@tool(
    "search_internal",
    "Search the internal channel: one organisation own reviews, notes and ledger extracts. "
    "Use it to find what was recorded inside the organisation about a field. Do not use it "
    "for published sector research (use search_web). Hits carry the same provenance as the "
    "published channel. A search that matches nothing returns an empty list, which is an "
    "answer and not an error.",
    {"query": Annotated[str, "What to search for, in plain words."]},
)
async def search_internal(args):
    return results(str(args.get("query", "")), "internal")


@tool(
    "load_document",
    "Open one document from the research corpus by URL and return its full text and "
    "provenance. The URL must be one a search returned. Use it to read a source before "
    "quoting from it. Do not use it to search. A URL outside the corpus is refused, with "
    "what to do instead.",
    {"url": Annotated[str, "A URL returned by search_web or search_internal."]},
)
async def load_document(args):
    url = str(args.get("url", "")).strip()
    doc = BY_URL.get(url)
    if doc is None:
        return failed(
            "validation", attempted=url,
            impact="This source contributes nothing until it is replaced.",
            alternatives=["Open a URL that search_web or search_internal actually returned"],
            # what is already usable and need not be redone: the documents that do open
            partial=[{"title": d["title"], "url": d["url"]} for d in DOCS],
        )
    return ok({"status": "success", **{k: v for k, v in doc.items() if k != "excerpt"}})


@tool(
    "verify_fact",
    "Check one claim against one corpus document: does the excerpt you were given really "
    "appear in that source, and what are its date and period? Use it while writing up, to "
    "confirm a finding before it goes into the report. Do not use it to find sources.",
    {"claim": Annotated[str, "The claim as it would appear in the report."],
     "url": Annotated[str, "The corpus URL the claim is attributed to."],
     "excerpt": Annotated[str, "The wording said to appear in that source."]},
)
async def verify_fact(args):
    url = str(args.get("url", "")).strip()
    doc = BY_URL.get(url)
    if doc is None:
        return failed("validation", attempted=url, impact="This claim stays unverified.",
                      alternatives=["Verify against a URL a search returned"])
    excerpt = " ".join(str(args.get("excerpt", "")).split()).lower()
    found = bool(excerpt) and excerpt in " ".join(doc["text"].split()).lower()
    return ok({"status": "success", "verified": found, "url": url,
               "claim": str(args.get("claim", "")),
               "publication_date": doc["publication_date"],
               "collection_period": doc["collection_period"],
               "note": "" if found else "That wording does not appear in this source."})


TOOLS = [search_web, search_internal, load_document, verify_fact]
SERVER = create_sdk_mcp_server(name="research", version="1.0.0", tools=TOOLS)

print(f"{'field':<11} {'channel':<9} {'published':<11} {'period':<19} publisher")
print("-" * 92)
for doc in DOCS:
    print(f"{doc['field']:<11} {doc['channel']:<9} {doc['publication_date']:<11} "
          f"{doc['collection_period']:<19} {doc['publisher']}")
print()
print(f"tools: {', '.join(t.name for t in TOOLS)}")

## 3. Call the tools directly

Three calls, with no model in the loop and nothing spent.

Watch the second and third carefully, because they look similar and are opposites. A search
that ran and matched nothing is a **success** with an empty list: the corpus answered, and the
answer is that there is nothing there. That is the visual-art gap the report will carry. A URL
that is not in the corpus is a **failure**, because nothing was answered at all. Merge the two
and an agent either retries a question that was already answered, or reports an outage as
"nothing found".

The refusal prints all six fields. That is worth watching rather than reading about: each one
is a question the caller would otherwise have to guess at.

In [ ]:
# A decorated tool is an SdkMcpTool rather than a plain function, so the handler is reached
# through .handler, and content is an array of typed blocks, so the payload travels as JSON
# in a text block.
async def call(name, args):
    result = await {t.name: t.handler for t in TOOLS}[name](args)
    return bool(result.get("is_error")), json.loads(result["content"][0]["text"])


def show(label, is_error, payload):
    print(f"{'REFUSED' if is_error else 'ok     '}  {label}")
    for key in ("status", "failure_type", "attempted_query", "coverage_impact", "note"):
        if payload.get(key):
            print(f"           {key}: {payload[key]}")
    for hit in payload.get("results", []):
        print(f"           results: {hit['title'][:44]:<44} {hit['url']}")
    for item in payload.get("partial_results", []):
        print(f"           partial_results: {item['title'][:44]:<44} {item['url']}")
    for route in payload.get("alternative_approaches", []):
        print(f"           alternative_approaches: {route}")
    print()


show("search_web for music, the published channel",
     *await call("search_web", {"query": "music and audio sessions"}))
show("search_internal for visual art, which it does not hold",
     *await call("search_internal", {"query": "visual art and design"}))
show("load_document on a URL nobody can check",
     *await call("load_document", {"url": "https://ai-creative-institute.test/2026/report"}))

## 4. The two searchers

Two roles, defined in code rather than in files on disk. `AgentDefinition` carries the whole
design in four fields, and each one does a different job:

| Field | Who reads it | What it decides |
|---|---|---|
| `description` | the **research agent**, when choosing whom to delegate to | routing. Say when to use this agent and when not to |
| `prompt` | the **subagent**, once it starts | the detail. It costs nothing until that agent runs |
| `tools` | the runtime | the boundary. Leave it out and the subagent inherits every tool in the run |
| `model` | the runtime | cost per role |

The agent's name is the dict key, not a field, and the fields that cross the wire are
camelCase, because they are shared with the TypeScript SDK. A snake_case keyword raises
`TypeError`, which the cell shows.

The two are partitioned by channel, not by subject, and the partition is mechanical: one holds
`search_web`, the other holds `search_internal`, and neither can reach the other's channel.
Partitioning the scope is what stops two subagents doing the same work twice, and doing it in
`tools` rather than in a prompt is what makes it hold. Each field gets one of each, so two
fields means four searchers running at once.

Both run on the smaller model, which keeps the lab cheap and repeatable. `model` is set
explicitly so the decision is visible.

In [ ]:
from claude_agent_sdk import AgentDefinition

CAP = 70

PROVENANCE = ("For every source give the claim, a verbatim excerpt, the URL, the publisher, "
              "the publication date and the period it measured. A source without a URL and a "
              "date is not usable.")

EMPTY = ("If the search ran and matched nothing, say so plainly and name the field and the "
         "channel you searched. An empty result is an answer about the world, not a failure "
         "of the tooling, and it must not be reported as one.")

ROSTER = {
    "web_searcher": AgentDefinition(
        description=("Searches the published channel, trade research and official statistics, "
                     "for one field and reports the sources with their provenance. Use it to "
                     "survey what has been reported publicly. Do not use it for the "
                     "organisation own records."),
        prompt=f"You search the published channel for one field.\n\n"
               f"Call search_web once for the field you were given, and nothing else.\n\n"
               f"{PROVENANCE}\n\n{EMPTY}\n\nAt most {CAP} words.",
        tools=["mcp__research__search_web"],
        model=MODEL,
        maxTurns=2,
    ),
    "document_searcher": AgentDefinition(
        description=("Searches the internal channel, the organisation own reviews, notes and "
                     "ledgers, and opens the document behind the best hit to quote from it. "
                     "Use it when a claim has to rest on internal wording. Do not use it for "
                     "published sector research."),
        prompt=f"You search the internal channel and quote from what you open.\n\n"
               f"Call search_internal once for the field you were given. If it returns a "
               f"hit, call load_document on the best one and quote from what you opened.\n\n"
               f"{PROVENANCE} Every quotation must be verbatim.\n\n{EMPTY}\n\n"
               f"If load_document refuses a URL, read the refusal: it says what to do "
               f"instead. Do not retry it, and do not invent a source you could not open. "
               f"At most {CAP} words.",
        tools=["mcp__research__search_internal", "mcp__research__load_document"],
        model=MODEL,
        maxTurns=3,
    ),
}

print(f"{'role':<19} {'tools':<52} maxTurns")
print("-" * 82)
for name, definition in ROSTER.items():
    print(f"{name:<19} "
          f"{', '.join(t.replace('mcp__research__', '') for t in definition.tools):<52} "
          f"{definition.maxTurns}")

# And the shape that does not exist. The community guide's example uses these keywords; the
# dataclass has description, prompt, tools, model, and camelCase for the rest.
print()
try:
    AgentDefinition(name="web_searcher", system_prompt="Search.", allowed_tools=["search_web"])
except TypeError as exc:
    print(f"TypeError: {exc}")

## 5. Stage one: four searchers at once

The first run spends. It is capped twice over, in turns and in money.

Three options on the run look like they do the same job and do not. `tools` is the
**session-wide** set of built-in tools, and a subagent's `tools` can only choose from it; MCP
tools are not built-ins, so it does not gate them at all. `allowed_tools` is what runs
**without asking**, which is approval and not restriction: it never narrows anything. Only
`AgentDefinition.tools` is a per-role boundary. So the research agent is given the spawning
tool and nothing else, and the channel split holds in the roster.

The brief asks for **all four searchers in one response**, because several spawn calls in a
single response run at the same time and the wait is the slowest of them rather than the sum.
The cell proves it by grouping the calls on the response they arrived in, never per call.

It also asks the research agent to report the findings and stop, rather than write anything
up. Writing up is the next stage, and keeping the two apart is what makes the handover visible.

`max_budget_usd` is a real stop rather than a warning: crossing it ends the run with an error
instead of a result, so `run` catches that. It is a safety net set above what a run should
cost, never a target.

In [ ]:
from collections import defaultdict

from claude_agent_sdk import (AssistantMessage, ClaudeAgentOptions, ResultError,
                              ResultMessage, ToolUseBlock, query)

SPAWN = ["Task", "Agent"]
RESEARCH_TOOLS = ["mcp__research__search_web", "mcp__research__search_internal",
                  "mcp__research__load_document"]
# A claude.ai login carries your organisation's connectors into the session, and
# setting_sources=[] does not exclude them: a connector is not a filesystem setting.
# Left on, a student signed in to a subscription gets MCP tools this lab never defined,
# in the middle of a lab about which tool the agent reaches for. A key does not load
# them, and neither does a setup-token, so this is the one line that makes the run the
# same on every credential.
NO_CONNECTORS = {"ENABLE_CLAUDEAI_MCP_SERVERS": "false"}

SPEND = []


async def run(prompt, *, budget, turns, **settings):
    options = ClaudeAgentOptions(model=MODEL, mcp_servers={"research": SERVER},
                                 setting_sources=[], env=NO_CONNECTORS,
                                 max_turns=turns, max_budget_usd=budget, **settings)
    calls, result = [], None
    try:
        async for message in query(prompt=prompt, options=options):
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    # ToolSearch is how the SDK loads a tool's schema on demand. It is
                    # plumbing, not a choice the model made, so it is not part of the count.
                    if isinstance(block, ToolUseBlock) and block.name != "ToolSearch":
                        calls.append({"name": block.name, "input": block.input,
                                      "response": message.message_id or "one response",
                                      # set when the message came from inside a subagent,
                                      # which is how a call is attributed to a role
                                      "from_subagent": message.parent_tool_use_id})
            elif isinstance(message, ResultMessage):
                result = message
    except ResultError as exc:
        print(f"the run stopped early: {exc}")
    if result is not None:
        SPEND.append(result.total_cost_usd or 0.0)
        print(f"cost: ${result.total_cost_usd:.4f}, turns: {result.num_turns}, "
              f"{result.duration_ms / 1000:.1f}s   (notebook total ${sum(SPEND):.4f})")
    return calls, result


RESEARCH_BRIEF = f"""GOAL
  {BRIEF['goal']}

FIELDS
{chr(10).join('  - ' + f['label'] for f in BRIEF['fields'])}

CRITERIA
{chr(10).join('  - ' + c for c in BRIEF['criteria'])}

HOW TO WORK
  - For each field, spawn a web_searcher and a document_searcher: four in total, all in the
    same response, so they run at the same time.
  - Then report what every searcher returned, each finding with its claim, verbatim excerpt,
    source URL, publisher, publication date and the period measured, and every empty search
    named with its field and channel.
  - Do not write the report. Another agent does that next, and it will only ever see what
    you hand it."""

research_calls, research_result = await run(
    RESEARCH_BRIEF, budget=0.20, turns=8,
    system_prompt="You lead a research team. You delegate, and each subagent starts with an "
                  "empty context, so whatever it needs must be in the prompt you send it.",
    tools=SPAWN,                            # availability: delegation and nothing else
    allowed_tools=SPAWN + RESEARCH_TOOLS,   # approval, which is a different thing
    agents=ROSTER,
)

spawns = [c for c in research_calls if c["name"] in ("Agent", "Task")]

print()
print("spawns, grouped by the response they arrived in:")
grouped = defaultdict(list)
for call_ in spawns:
    grouped[call_["response"]].append(call_["input"].get("subagent_type", "?"))
for index, kinds in enumerate(grouped.values(), 1):
    print(f"  response {index}: {', '.join(kinds)}"
          f"  ({'together' if len(kinds) > 1 else 'on its own'})")

print()
print(f"the stream reports the spawning tool as: {sorted({c['name'] for c in spawns})}")
print()
by_agent = defaultdict(list)
for call_ in research_calls:
    if call_["from_subagent"]:
        by_agent[call_["from_subagent"]].append(call_["name"].replace("mcp__research__", ""))
for index, used in enumerate(by_agent.values(), 1):
    print(f"  subagent {index} used: {', '.join(used)}")

FINDINGS = (research_result.result or "") if research_result else ""
print()
print("the findings, which are the only thing the next stage will receive:")
print(FINDINGS[:1200])

## 6. Stage two: chain the findings into a formatter

The second run is a different agent. It has no roster, it cannot spawn anything, and its only
tool is `verify_fact`, so it can check a finding but it cannot go and look for one.

It also starts with an empty context. It has not seen the brief, the searchers, or anything
the first run did, and the only thing it will ever know about the research is the text in
`FINDINGS` that the cell pastes into its prompt. That is the rule that governs every
delegation in this course, and chaining two runs is the plainest way to see it: leave something
out of the prompt and it is not there at all.

`output_format` makes the answer a validated object rather than prose, and it arrives on
`ResultMessage.structured_output`. The harness checks it against the JSON schema and stops
there, so the rules a schema cannot express run in `model_validate`: a finding with no source
URL and no excerpt is not established, and a contested entry needs two values from two
different sources, because two numbers from one publisher are not a disagreement.

Note what is deliberately **not** a rule. Visual art comes back both well established and a
coverage gap, and that is correct rather than a contradiction: the published channel answered
and the internal one did not. A gap names a field **and a channel**, which is why `Gap` carries
both. An earlier draft of this lab forbade the overlap, and the first live run was refused by
its own validator. A contract has to describe the research the run can actually do.

Read the three sections against the table in the opening cell. Contested is the part that
would have been lost if the formatter had picked the more credible publisher, or averaged two
numbers into one nobody reported: the two session-hour figures differ because one counted
nineteen studios over eight months and excluded home studios, and the other counted every
project in a quarter and included them.

In [ ]:
from pydantic import BaseModel, Field, model_validator


class Claim(BaseModel):
    field_id: str
    claim: str
    excerpt: str
    source_name: str
    source_url: str
    publication_date: str = ""
    collection_period: str = ""


class ContestedValue(BaseModel):
    value: str
    source_name: str
    source_url: str
    collection_period: str = ""


class Contested(BaseModel):
    claim: str
    values: list[ContestedValue]
    possible_explanation: str = ""


class Gap(BaseModel):
    field_id: str
    channel: str = ""
    reason: str


class ResearchReport(BaseModel):
    topic: str
    well_established: list[Claim] = Field(default_factory=list)
    contested: list[Contested] = Field(default_factory=list)
    coverage_gaps: list[Gap] = Field(default_factory=list)

    @model_validator(mode="after")
    def _rules_the_schema_cannot_express(self):
        thin = [c.claim[:40] for c in self.well_established if not (c.source_url and c.excerpt)]
        if thin:
            raise ValueError(f"established claims without a url and an excerpt: {thin}")
        for entry in self.contested:
            # Two values are not a disagreement unless they came from two sources. There
            # is deliberately no rule against a field appearing in both well_established
            # and coverage_gaps: visual art is in both here, because the published
            # channel answered and the internal one did not.
            if len({v.source_url for v in entry.values}) < 2:
                raise ValueError(
                    f"contested entry {entry.claim[:40]!r} does not keep two sources")
        return self


FORMATTER = """You write a research report from findings you are given.

Confirm a finding against its source with verify_fact before it goes into well_established.

Write three sections and nothing else:
  well_established   findings that stand, each with its verbatim excerpt, URL and date.
  contested          disagreements, every value kept with its publisher and period, plus
                     what might explain the difference. Never choose between them and never
                     average them.
  coverage_gaps      a field and the channel that returned nothing, and why that leaves
                     a gap. A field whose other channel did answer still belongs here.

Every field_id must be one of the ids you were given, spelled exactly as written.

You cannot search. If a finding you would need is missing, record it as a coverage gap."""

report_calls, report_result = await run(
    f"TOPIC\n  {BRIEF['topic']}\n\n"
    f"FIELD IDS, to be used exactly as written\n"
    f"{chr(10).join('  ' + f['id'] + '   ' + f['label'] for f in BRIEF['fields'])}\n\n"
    f"FINDINGS FROM THE RESEARCH RUN\n\n{FINDINGS}",
    budget=0.10, turns=4,
    system_prompt=FORMATTER,
    tools=[],                                          # it cannot spawn anything
    allowed_tools=["mcp__research__verify_fact"],      # and this is all it can call
    output_format={"type": "json_schema", "schema": ResearchReport.model_json_schema()},
)

payload = report_result.structured_output if report_result else None
print()
if payload is None:
    print("no structured output on this run, so there is nothing to validate.")
else:
    report = ResearchReport.model_validate(payload)
    print(f"topic: {report.topic}")
    print()
    print(f"WELL ESTABLISHED ({len(report.well_established)})")
    for claim in report.well_established:
        print(f"  [{claim.field_id}] {claim.claim}")
        print(f"      \"{claim.excerpt[:92]}\"")
        print(f"      {claim.source_name}, {claim.source_url} {claim.publication_date}")
    print()
    print(f"CONTESTED ({len(report.contested)})")
    for entry in report.contested:
        print(f"  {entry.claim}")
        for value in entry.values:
            print(f"      {value.value}  ({value.source_name}, {value.collection_period})")
        print(f"      why they might differ: {entry.possible_explanation}")
    print()
    print(f"COVERAGE GAPS ({len(report.coverage_gaps)})")
    for gap in report.coverage_gaps:
        print(f"  [{gap.field_id}/{gap.channel or 'any channel'}] {gap.reason}")

    written = OUT / "report.json"
    written.write_text(json.dumps(report.model_dump(), indent=2) + "\n", encoding="utf-8")
    print()
    print(f"wrote {written.relative_to(LAB)}")

print()
print(f"this notebook spent ${sum(SPEND):.4f} over {len(SPEND)} runs")

## What you built

| Stage | Where it was built |
|---|---|
| Research brief | Section 1 |
| MCP server, four tools on two channels | Sections 2 and 3 |
| Six-field failure contract | Sections 2 and 3 |
| Two searchers, partitioned by channel | Section 4 |
| Research agent, four spawns in one response | Section 5 |
| Findings chained into the formatter's prompt | Section 6 |
| ResearchReport: established, contested, gaps | Section 6 |

Four decisions to carry out of this lab.

Given work that splits cleanly, spawn it in **one response**. Several spawn calls in a single
response run at the same time, and the wait is the slowest of them rather than the sum. Spread
the same calls across turns and nothing is parallel.

Given a handover, put the findings **in the prompt**. The next agent's context starts empty,
whether it is a subagent or a chained run, so anything you leave out is not there at all.

Given a subagent that must not do something, reach for `AgentDefinition.tools`. An allowlist
approves and does not restrict, and `tools` on the run is session-wide; the per-role list is
the only boundary there is.

Given two sources that disagree, keep both. A report that resolves a conflict silently has told
the reader less than the research found, and a search that matched nothing is a finding about
the world rather than a fault in the tooling.